# Detección binaria de DoS en MQTT — MQTT_UAD

Este notebook entrena **tres modelos binarios de DoS para MQTT sin cifrar**:
XGBoost, LSTM Autoencoder e híbrido LSTM + XGBoost, todos con etiqueta
`normal` o `ataque`, donde `ataque` significa exclusivamente DoS. Cada modelo tiene reporte, confusion matrix, accuracy,
balanced accuracy, macro-F1 y ROC-AUC.
Sus hiperparámetros se optimizan con estudios Optuna independientes y
3-fold grouped cross-validation interna; test queda reservado para la
evaluación final.

Se usa únicamente `DoS.csv`; MitM e Intrusion quedan fuera del entrenamiento,
la validación y la exportación. Se conservan las rutas de entrada y salida.
Las features combinan red/transporte y campos MQTT numéricos, entropía de payload
y estructura de topic. IP, client_id, credenciales, puertos efímeros y tiempos
absolutos/relativos de captura solo sirven de contexto o se excluyen.

La captura admite todas las tramas. El agente clasifica frames IPv4/TCP con
endpoints completos; el notebook aplica esa misma selección. Los CSV no tienen
tcp.stream: se aproximan sesiones mediante endpoints y CONNECT.
Las particiones separan conexiones completas y sus dos direcciones; las
ventanas no cruzan archivos, conexiones, direcciones ni particiones.
No es una prueba de generalización a capturas independientes ni a MQTTS.

In [ ]:
import os
import json
import math
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
import joblib
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             balanced_accuracy_score, f1_score, roc_auc_score)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    plt = sns = None  # Los reportes numéricos siguen funcionando sin librerías de plots.

SEED = 42
SEQ_LEN = 10
N_SPLITS = 5
INNER_SPLITS = 3
NAN_FILL = -1.0
OPTUNA_TRIALS_XGB = 30
OPTUNA_TRIALS_LSTM = 15
OPTUNA_TRIALS_HYBRID = 30
optuna.logging.set_verbosity(optuna.logging.WARNING)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
XGB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Torch: {DEVICE}; XGBoost: {XGB_DEVICE}')


## 1. Carga y contexto de las capturas

No se asignan etiquetas de ataque a un archivo completo: se conserva `type` por frame.


In [ ]:
FILES = {
    'DoS': '/kaggle/input/datasets/alejandropenagos79/dataset/DoS.csv',
}
TARGET_COL = 'type'

LABEL_MAP = {'dos': 'DoS', 'normal': 'normal'}
FLOW_COLUMNS = ['ip.src', 'tcp.srcport', 'ip.dst', 'tcp.dstport']

def load_and_merge(files):
    frames = []
    for capture, path in files.items():
        data = pd.read_csv(path, low_memory=False, dtype={
            'mqtt.msg': 'string', 'mqtt.topic': 'string',
            'ip.src': 'string', 'ip.dst': 'string',
        })
        if 'Type' in data.columns and TARGET_COL not in data.columns:
            data = data.rename(columns={'Type': TARGET_COL})
        required = FLOW_COLUMNS + [
            'frame.time_epoch', 'frame.time_delta', 'frame.time_delta_displayed',
            'frame.len', 'frame.cap_len', 'mqtt.msgtype', TARGET_COL,
        ]
        missing = set(required) - set(data.columns)
        if missing:
            raise ValueError(f'{capture}: faltan columnas {sorted(missing)}')
        labels = data[TARGET_COL].astype(str).str.strip().str.lower()
        if not labels.isin(LABEL_MAP).all():
            raise ValueError(f'{capture}: etiquetas desconocidas {labels[~labels.isin(LABEL_MAP)].unique()}')
        data[TARGET_COL] = labels.map(LABEL_MAP)
        for col in ('tcp.srcport', 'tcp.dstport', 'frame.time_epoch', 'mqtt.msgtype'):
            data[col] = pd.to_numeric(data[col], errors='coerce')
        print(f'{capture}: todas las tramas: {data[TARGET_COL].value_counts().to_dict()}')
        # Misma población que Agent: IPv4/TCP con endpoints completos.
        before = len(data)
        data = data.dropna(subset=FLOW_COLUMNS + ['frame.time_epoch'])
        valid = (
            data['ip.src'].str.strip().ne('') & data['ip.dst'].str.strip().ne('')
            & data['tcp.srcport'].between(1, 65535)
            & data['tcp.dstport'].between(1, 65535)
            & data['tcp.srcport'].mod(1).eq(0) & data['tcp.dstport'].mod(1).eq(0)
            & np.isfinite(data['frame.time_epoch'])
        )
        data = data.loc[valid].copy()
        data['_original_row'] = data.index
        print(f'{capture}: excluidas por contexto inválido: {before - len(data)}')
        data = data.sort_values('frame.time_epoch', kind='stable')
        data['_capture'] = capture
        endpoints = [
            tuple(sorted(((str(src), int(sport)), (str(dst), int(dport)))))
            for src, sport, dst, dport in data[FLOW_COLUMNS].itertuples(index=False, name=None)
        ]
        data['_endpoints'] = pd.Series(endpoints, index=data.index, dtype=object)
        # CSV no tiene tcp.stream: un CONNECT nuevo separa sesiones con los mismos endpoints.
        # Sin SYN/FIN no podemos reconstruir todas las sesiones TCP con certeza.
        connect = data['mqtt.msgtype'].eq(1).astype(int)
        session = connect.groupby(data['_endpoints'], sort=False).cumsum()
        data['_connection'] = [
            (capture, endpoint, int(number))
            for endpoint, number in zip(endpoints, session)
        ]
        data['_direction'] = [
            (connection, str(src), int(sport), str(dst), int(dport))
            for connection, (src, sport, dst, dport)
            in zip(data['_connection'], data[FLOW_COLUMNS].itertuples(index=False, name=None))
        ]
        print(f'{capture}: IPv4/TCP seleccionadas: {data[TARGET_COL].value_counts().to_dict()}')
        frames.append(data)
    merged = pd.concat(frames, ignore_index=True)
    if merged.empty:
        raise ValueError('No hay frames IPv4/TCP con endpoints completos')
    return merged

df_raw = load_and_merge(FILES)
print(f'Dataset IPv4/TCP seleccionado: {df_raw.shape}')
print(df_raw[TARGET_COL].value_counts())


## 2. Features MQTT y diagnostico de columnas

Se conservan **13 features**: red/transporte, campos MQTT con senal y derivadas de
entropia y ratio. Antes de construir la matriz se imprime el diagnostico por columna
cruda. Las tablas siguientes resumen, por grupo, que se conserva y que se descarta.

### Criterios de descarte

| Motivo | Criterio sobre los datos | Tratamiento |
|---|---|---|
| Columna vacia | 100% de valores nulos | Se descarta |
| Metadato de conexion | Mas del 99% nula y constante (solo aparece en CONNECT/CONNACK) | Se descarta |
| Identidad | IP, MAC o `client_id` | Se descarta |
| Fuga de captura | Reloj absoluto/relativo u orden de trama | Se descarta |
| Puerto efimero | `tcp.srcport`/`tcp.dstport` | Se resume en banderas `fe_is_*` |
| Duplicado exacto | Columna igual a otra en el 100% de las tramas | Se descarta |
| Derivables | Contenido ya codificado en `mqtt.hdrflags` | Se descarta |
| Texto crudo | `mqtt.msg`/`mqtt.topic` no numericos | Se resume en entropia/profundidad |
| Constante | Un unico valor en todas las tramas | Se descarta |

### Features conservadas (13)

| Grupo | Columnas | Que aportan |
|---|---|---|
| Red | `frame.len`, `frame.time_delta` | Tamano de trama y ritmo de envio |
| Red derivadas | `fe_bytes_per_sec`, `fe_is_standard_mqtt_port`, `fe_is_broker_to_client`, `fe_log_len` | Tasa de bytes, puerto MQTT y escala logaritmica |
| MQTT | `mqtt.hdrflags`, `mqtt.len`, `mqtt.topic_len` | Tipo de paquete, longitud MQTT y del topic |
| MQTT derivadas | `fe_msg_entropy`, `fe_topic_entropy`, `fe_topic_depth`, `fe_payload_ratio` | Entropia y estructura del topic, ratio de payload |

### Columnas obtenidas con feature engineering (8)

Estas columnas no existen directamente en el CSV. Se calculan a partir de los
campos capturados antes de construir la matriz final de 13 features.

| Columna generada | Fuente | Cálculo / significado |
|---|---|---|
| `fe_bytes_per_sec` | `frame.len`, `frame.time_delta` | `frame.len / (frame.time_delta + 1e-6)`; aproxima la intensidad instantánea del tráfico |
| `fe_is_standard_mqtt_port` | `tcp.dstport` | Vale 1 cuando el destino es 1883 o 8883; en otro caso vale 0 |
| `fe_is_broker_to_client` | `tcp.srcport` | Vale 1 cuando el origen es 1883 o 8883; identifica la dirección broker → cliente |
| `fe_log_len` | `frame.len` | `log1p(frame.len)`; reduce la escala de tamaños grandes |
| `fe_msg_entropy` | `mqtt.msg` | Entropía de Shannon del payload representado como texto |
| `fe_topic_entropy` | `mqtt.topic` | Entropía de Shannon del nombre del topic |
| `fe_topic_depth` | `mqtt.topic` | Número de niveles del topic: cantidad de `/` más uno |
| `fe_payload_ratio` | `mqtt.len`, `frame.len` | `mqtt.len / (frame.len + 1e-5)`; proporción MQTT respecto al tamaño total |

Las cinco columnas crudas conservadas son `frame.len`, `frame.time_delta`,
`mqtt.hdrflags`, `mqtt.len` y `mqtt.topic_len`. Junto con las ocho anteriores
forman exactamente las 13 columnas usadas por XGBoost, LSTM e híbrido.

### Columnas descartadas

| Columna(s) | Semantica | Motivo |
|---|---|---|
| `frame.time_invalid`, `frame.coloring_rule.name`, `frame.coloring_rule.string`, `frame.comment`, `frame.comment.expert`, `frame.file_off`, `frame.incomplete`, `frame.interface_id`, `frame.interface_name`, `frame.link_nr`, `frame.md5_hash` | Metadatos internos de tshark | 100% nulas |
| `mqtt.username`, `mqtt.username_len`, `mqtt.passwd`, `mqtt.passwd_len`, `mqtt.willmsg`, `mqtt.willmsg_len`, `mqtt.willtopic`, `mqtt.willtopic_len` | Credenciales y testamento MQTT | 100% nulas |
| `mqtt.sub.qos`, `mqtt.suback.qos` | QoS de SUBSCRIBE y SUBACK | Practicamente 100% nulas |
| `mqtt.conack.flags`, `mqtt.conack.flags.reserved`, `mqtt.conack.flags.sp`, `mqtt.conack.val`, `mqtt.conflag.cleansess`, `mqtt.conflag.passwd`, `mqtt.conflag.qos`, `mqtt.conflag.reserved`, `mqtt.conflag.retain`, `mqtt.conflag.uname`, `mqtt.conflag.willflag`, `mqtt.conflags`, `mqtt.kalive`, `mqtt.proto_len`, `mqtt.protoname`, `mqtt.ver` | Parametros de CONNECT y CONNACK | Mas del 99% nulas |
| `mqtt.msgid` | Identificador de paquete PUBLISH con QoS mayor que 0 | Mas del 99% nula |
| `mqtt.clientid`, `mqtt.clientid_len` | Identidad de cliente en CONNECT | Identidad |
| `ip.src`, `ip.dst`, `eth.src`, `eth.dst` | Identidad de host y de MAC | No describe comportamiento |
| `frame.time_epoch`, `frame.time_relative`, `frame.number` | Reloj y orden de captura | Fuga entre capturas |
| `tcp.srcport`, `tcp.dstport` | Puertos TCP efectivos | Leakage de puertos efimeros; solo banderas |
| `frame.cap_len` | Bytes capturados de la trama | Igual a `frame.len` en el 100% |
| `frame.time_delta_displayed` | Delta mostrado de tiempo | Igual a `frame.time_delta` en el 100% |
| `mqtt.msgtype`, `mqtt.qos`, `mqtt.retain`, `mqtt.dupflag` | Tipo y flags de PUBLISH | Derivables de `mqtt.hdrflags` |
| `mqtt.msg`, `mqtt.topic` | Payload y topic publicados | Texto crudo; se resumen en derivadas |
| `frame.encap_type`, `frame.ignored`, `frame.marked`, `frame.offset_shift` | Banderas internas de captura | Constantes, sin varianza |

La validacion de particiones (seccion 3) calcula las etiquetas de ventana por fold
una sola vez, de modo que no recorre el dataset en cada combinacion.

In [ ]:
def entropia(texto):
    if pd.isna(texto) or not str(texto):
        return 0.0
    text = str(texto)
    probs = [count / len(text) for count in Counter(text).values()]
    return -sum(p * math.log2(p) for p in probs)


# Features crudas conservadas.
KEEP_RAW = ['frame.len', 'frame.time_delta', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.topic_len']
# Columnas que no se conservan crudas pero alimentan una derivada.
DERIVED_SOURCES = {
    'tcp.dstport': 'fe_is_standard_mqtt_port',
    'tcp.srcport': 'fe_is_broker_to_client',
    'mqtt.msg': 'fe_msg_entropy',
    'mqtt.topic': 'fe_topic_entropy y fe_topic_depth',
}
# Features derivadas generadas y su destino.
DERIVED_FEATURES = {
    'fe_bytes_per_sec': ('CONSERVA', 'Tasa de bytes (frame.len / frame.time_delta).'),
    'fe_is_standard_mqtt_port': ('CONSERVA', 'Destino en 1883 o 8883.'),
    'fe_is_broker_to_client': ('CONSERVA', 'Origen en 1883 o 8883.'),
    'fe_log_len': ('CONSERVA', 'log1p(frame.len); escala para el LSTM.'),
    'fe_msg_entropy': ('CONSERVA', 'Entropia del payload.'),
    'fe_topic_entropy': ('CONSERVA', 'Entropia del topic.'),
    'fe_topic_depth': ('CONSERVA', 'Niveles del topic.'),
    'fe_payload_ratio': ('CONSERVA', 'Ratio mqtt.len / frame.len.'),
    'fe_cap_ratio': ('DESCARTA', 'Constante 1.0 con frame.cap_len igual a frame.len.'),
    'fe_anon_connect': ('DESCARTA', 'Casi siempre 0; no separa normal de ataque.'),
}
# Motivos de descarte que no dependen de la tasa de nulos.
RAZONES = {
    'frame.time_delta_displayed': 'Duplica frame.time_delta',
    'frame.time_epoch': 'Fuga de captura (reloj absoluto)',
    'frame.time_relative': 'Fuga de captura (reloj relativo)',
    'frame.number': 'Fuga de captura (orden de trama)',
    'frame.cap_len': 'Duplica frame.len',
    'frame.encap_type': 'Constante',
    'frame.ignored': 'Constante',
    'frame.marked': 'Constante',
    'frame.offset_shift': 'Constante',
    'ip.src': 'Identidad de host',
    'ip.dst': 'Identidad de host',
    'eth.src': 'Identidad de capa 2',
    'eth.dst': 'Identidad de capa 2',
    'mqtt.clientid': 'Identidad de cliente',
    'mqtt.msgtype': 'Derivable de mqtt.hdrflags',
    'mqtt.qos': 'Derivable de mqtt.hdrflags',
    'mqtt.retain': 'Derivable de mqtt.hdrflags',
    'mqtt.dupflag': 'Derivable de mqtt.hdrflags',
    'mqtt.msg': 'Texto crudo (se resume en entropia)',
    'mqtt.topic': 'Texto crudo (se resume en entropia)',
    'tcp.srcport': 'Puerto efimero (solo bandera)',
    'tcp.dstport': 'Puerto efimero (solo bandera)',
}
# Columnas internas que no son features.
INTERNAS = ('type', '_capture', '_original_row', '_endpoints', '_connection', '_direction')

# Catálogo explícito de las 13 columnas finales.
RAW_FEATURE_COLUMNS = [
    'frame.len', 'frame.time_delta', 'mqtt.hdrflags',
    'mqtt.len', 'mqtt.topic_len',
]
ENGINEERED_FEATURE_COLUMNS = [
    'fe_bytes_per_sec', 'fe_is_standard_mqtt_port',
    'fe_is_broker_to_client', 'fe_log_len',
    'fe_msg_entropy', 'fe_topic_entropy',
    'fe_topic_depth', 'fe_payload_ratio',
]

# Orden final de la matriz del modelo; no cambiar sin reentrenar.
FEATURE_COLUMNS = [
    'frame.len', 'frame.time_delta',
    'fe_bytes_per_sec', 'fe_is_standard_mqtt_port', 'fe_is_broker_to_client', 'fe_log_len',
    'mqtt.hdrflags', 'mqtt.len', 'mqtt.topic_len',
    'fe_msg_entropy', 'fe_topic_entropy', 'fe_topic_depth', 'fe_payload_ratio',
]
assert len(FEATURE_COLUMNS) == 13
assert set(FEATURE_COLUMNS) == set(RAW_FEATURE_COLUMNS + ENGINEERED_FEATURE_COLUMNS)

In [ ]:
def diagnostico_columnas(df):
    """Clasifica cada columna cruda: nulos, cardinalidad, decision y motivo."""
    filas = []
    for col in df.columns:
        if col in INTERNAS:
            continue
        nulos = float(df[col].isna().mean())
        distintos = int(df[col].nunique(dropna=True))
        if col in KEEP_RAW:
            decision, motivo = 'CONSERVA', 'Feature cruda con senal'
        elif col in DERIVED_SOURCES:
            decision, motivo = 'DERIVA', 'Resume en ' + DERIVED_SOURCES[col]
        elif col in RAZONES:
            decision, motivo = 'DESCARTA', RAZONES[col]
        elif nulos == 1.0:
            decision, motivo = 'DESCARTA', 'Vacia (100% nulos)'
        elif nulos >= 0.99:
            decision, motivo = 'DESCARTA', 'Metadato esporadico (>99% nula)'
        else:
            decision, motivo = 'DESCARTA', 'Sin senal'
        filas.append({'columna': col, 'nulos%': round(100.0 * nulos, 2),
                      'distintos': distintos, 'decision': decision, 'motivo': motivo})
    return pd.DataFrame(filas)


def imprimir_tabla(tabla, columnas):
    """Imprime un DataFrame en columnas de ancho fijo alineadas."""
    anchos = {c: max(len(c), max(len(str(v)) for v in tabla[c])) for c in columnas}
    print('  '.join(c.ljust(anchos[c]) for c in columnas))
    print('  '.join('-' * anchos[c] for c in columnas))
    for _, fila in tabla.iterrows():
        print('  '.join(str(fila[c]).ljust(anchos[c]) for c in columnas))


diagnostico = diagnostico_columnas(df_raw)
orden = {'CONSERVA': 0, 'DERIVA': 1, 'DESCARTA': 2}
diagnostico['_orden'] = diagnostico['decision'].map(orden)
diagnostico = (diagnostico.sort_values(['_orden', 'nulos%'], ascending=[True, False])
                          .drop(columns='_orden').reset_index(drop=True))
conteo = diagnostico['decision'].value_counts()
print('DIAGNOSTICO DE COLUMNAS CRUDAS')
print(f"evaluadas={len(diagnostico)}  conserva={conteo.get('CONSERVA', 0)}  "
      f"deriva={conteo.get('DERIVA', 0)}  descarta={conteo.get('DESCARTA', 0)}")
print()
imprimir_tabla(diagnostico, ['columna', 'nulos%', 'distintos', 'decision', 'motivo'])
print()
print('FEATURES DERIVADAS')
derivadas = pd.DataFrame([{'derivada': nombre, 'decision': datos[0], 'motivo': datos[1]}
                          for nombre, datos in DERIVED_FEATURES.items()])
imprimir_tabla(derivadas, ['derivada', 'decision', 'motivo'])

In [ ]:
NETWORK_RAW = ['frame.len', 'frame.time_delta']
MQTT_NUMERIC_COLUMNS = ['mqtt.hdrflags', 'mqtt.len', 'mqtt.topic_len']


def build_network_features(X_full):
    """Red/transporte sin puertos crudos ni reloj de captura."""
    base = X_full[NETWORK_RAW].apply(pd.to_numeric, errors='coerce').fillna(NAN_FILL).copy()
    base['fe_bytes_per_sec'] = base['frame.len'] / (base['frame.time_delta'] + 1e-6)
    dst = pd.to_numeric(X_full['tcp.dstport'], errors='coerce')
    src = pd.to_numeric(X_full['tcp.srcport'], errors='coerce')
    base['fe_is_standard_mqtt_port'] = dst.isin([1883, 8883]).astype(int)
    base['fe_is_broker_to_client'] = src.isin([1883, 8883]).astype(int)
    base['fe_log_len'] = np.log1p(base['frame.len'])
    return base


def build_mqtt_features(df):
    """Matriz compacta MQTT: sin identidades, puertos crudos ni reloj de captura."""
    result = build_network_features(df)

    def number(value):
        if isinstance(value, str) and value.lower().startswith('0x'):
            try:
                return int(value, 16)
            except ValueError:
                return np.nan
        return value

    for col in MQTT_NUMERIC_COLUMNS:
        series = df[col] if col in df else pd.Series(np.nan, index=df.index)
        result[col] = pd.to_numeric(series.map(number), errors='coerce').fillna(NAN_FILL)
    msg = df['mqtt.msg'] if 'mqtt.msg' in df else pd.Series(None, index=df.index, dtype=object)
    topic = df['mqtt.topic'] if 'mqtt.topic' in df else pd.Series(None, index=df.index, dtype=object)
    result['fe_msg_entropy'] = msg.apply(entropia)
    result['fe_topic_entropy'] = topic.apply(entropia)
    result['fe_topic_depth'] = topic.apply(
        lambda value: 0 if pd.isna(value) or not str(value) else str(value).count('/') + 1)
    result['fe_payload_ratio'] = result['mqtt.len'].clip(lower=0) / (
        result['frame.len'].clip(lower=0) + 1e-5)
    return result.replace([np.inf, -np.inf], NAN_FILL).fillna(NAN_FILL)


def build_feature_matrix(df):
    """Matriz final en el orden de FEATURE_COLUMNS."""
    return build_mqtt_features(df)[FEATURE_COLUMNS]

In [ ]:
X = build_feature_matrix(df_raw)
le_target = LabelEncoder().fit(df_raw[TARGET_COL])
CLASS_NAMES = le_target.classes_
y = pd.Series(le_target.transform(df_raw[TARGET_COL]), index=df_raw.index)
normal_idx = int(le_target.transform(['normal'])[0])
assert all(pd.api.types.is_numeric_dtype(X[col]) for col in X)
assert np.isfinite(X.to_numpy()).all()
print(f'Matriz final: {X.shape}; clases: {list(CLASS_NAMES)}')
print(f'Columnas crudas conservadas ({len(RAW_FEATURE_COLUMNS)}):', RAW_FEATURE_COLUMNS)
print(f'Columnas obtenidas con feature engineering ({len(ENGINEERED_FEATURE_COLUMNS)}):',
      ENGINEERED_FEATURE_COLUMNS)
feature_catalog = pd.DataFrame({
    'columna': FEATURE_COLUMNS,
    'origen': [
        'cruda conservada' if name in RAW_FEATURE_COLUMNS else 'feature engineering'
        for name in FEATURE_COLUMNS
    ],
})
print('\nCatálogo de las 13 features en orden de entrenamiento:')
print(feature_catalog.to_string(index=False))
print('\nVista previa de la matriz final:')
print(X.head())

## 3. Development / test externo por conexión

Se conserva un test externo bloqueado. Los antiguos train y validation forman
development, donde Optuna ejecuta 3-fold grouped cross-validation. Se exige
cobertura de clases con ventanas completas y ninguna conexión cruza folds.


In [ ]:
connection_ids = pd.factorize(df_raw['_connection'], sort=False)[0]
direction_ids = pd.factorize(df_raw['_direction'], sort=False)[0]


def sequence_indices(indices, seq_length=SEQ_LEN):
    """Ventanas por direccion y particion; el agente comienza con buffer de 11."""
    selected = np.asarray(indices, dtype=int)
    windows = []
    grouped = pd.Series(direction_ids[selected], index=selected)
    for members in grouped.groupby(grouped, sort=False).groups.values():
        rows = np.asarray(members, dtype=int)
        rows = rows[np.argsort(df_raw.iloc[rows]['frame.time_epoch'].to_numpy(), kind='stable')]
        # La primera prediccion del Agent exige 11 filas; clasifica las ultimas 10.
        for start in range(1, len(rows) - seq_length + 1):
            windows.append(rows[start:start + seq_length])
    return np.asarray(windows, dtype=int).reshape(-1, seq_length)


splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = [test for _, test in splitter.split(X, y, groups=connection_ids)]

# Etiquetas y conteo de ventanas por fold: se calculan una sola vez para que la
# seleccion de particiones no recorra el dataset en cada combinacion.
ALL_CLASSES = set(range(len(CLASS_NAMES)))
fold_windows = [sequence_indices(fold) for fold in folds]
fold_counts = [len(rows) for rows in fold_windows]
fold_labels = [set(y.iloc[rows[:, -1]].tolist()) for rows in fold_windows]


def covered(fold_ids):
    labels = set()
    for index in fold_ids:
        labels |= fold_labels[index]
    return labels == ALL_CLASSES and sum(fold_counts[index] for index in fold_ids) > 0


partitions = None
for test_fold in range(N_SPLITS):
    for val_fold in range(N_SPLITS):
        if test_fold == val_fold:
            continue
        train_folds = [i for i in range(N_SPLITS) if i not in (test_fold, val_fold)]
        if covered(train_folds) and covered([val_fold]) and covered([test_fold]):
            partitions = {
                'train': np.concatenate([folds[i] for i in train_folds]),
                'val': folds[val_fold],
                'test': folds[test_fold],
            }
            break
    if partitions is not None:
        break
if partitions is None:
    raise ValueError('No hay particiones de conexiones con todas las clases evaluables. '
                     'No cambiar a split aleatorio de frames: revisar cobertura del dataset.')
train_idx, val_idx, test_idx = (partitions[name] for name in ('train', 'val', 'test'))
development_idx = np.concatenate([train_idx, val_idx])
for left, right in (('train', 'val'), ('train', 'test'), ('val', 'test')):
    assert not set(connection_ids[partitions[left]]) & set(connection_ids[partitions[right]])

# Las ventanas de cada particion se calculan una vez y se reutilizan.
window_rows = {name: sequence_indices(rows) for name, rows in partitions.items()}
for name, rows in partitions.items():
    windows = window_rows[name]
    print(name, 'frames:', len(rows), 'clases:', df_raw.iloc[rows][TARGET_COL].value_counts().to_dict(),
          'ventanas:', len(windows),
          'clases evaluables:', df_raw.iloc[windows[:, -1]][TARGET_COL].value_counts().to_dict())
    print(f'{name}: cobertura de ventanas={len(windows) / len(rows):.2%}')

inner_splitter = StratifiedGroupKFold(
    n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
inner_partitions = [
    (development_idx[inner_train], development_idx[inner_val])
    for inner_train, inner_val in inner_splitter.split(
        X.iloc[development_idx], y.iloc[development_idx],
        groups=connection_ids[development_idx])
]
for fold_id, (inner_train, inner_val) in enumerate(inner_partitions):
    assert not set(connection_ids[inner_train]) & set(connection_ids[inner_val])
    train_windows = sequence_indices(inner_train)
    val_windows = sequence_indices(inner_val)
    if (set(y.iloc[train_windows[:, -1]]) != ALL_CLASSES
            or set(y.iloc[val_windows[:, -1]]) != ALL_CLASSES):
        raise ValueError(f'Fold interno {fold_id} sin cobertura de clases en ventanas')
    print(f'inner fold {fold_id}: train={len(inner_train)} frames, '
          f'validation={len(inner_val)} frames')
print('Clases:', list(CLASS_NAMES))

## 4. Helpers y medidas por modelo


In [ ]:
BINARY_NAMES = ['normal', 'ataque']
y_bin = pd.Series((y.to_numpy() != normal_idx).astype(int), index=y.index)
# LabelEncoder con orden fijo 0 = normal, 1 = ataque para la exportación.
le_binary = LabelEncoder().fit(BINARY_NAMES)
le_binary.classes_ = np.asarray(BINARY_NAMES)


def suggest_xgb_params(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 12.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 20.0, log=True),
    }


def get_xgb_model(params=None):
    return XGBClassifier(objective='binary:logistic', tree_method='hist',
                         device=XGB_DEVICE, random_state=SEED, eval_metric='logloss',
                         **(params or {}))

def fit_classifier(features, labels, params=None):
    model = get_xgb_model(params)
    model.fit(features, labels, sample_weight=compute_sample_weight('balanced', labels))
    return model

metrics = {}

def evaluate(name, actual, predicted, probabilities, names):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    labels = np.arange(len(names))
    print(f'\n{name}')
    report = classification_report(actual, predicted, labels=labels,
                                   target_names=list(names), digits=4, zero_division=0)
    print(report)
    cm = confusion_matrix(actual, predicted, labels=labels)
    result = {
        'accuracy': float(accuracy_score(actual, predicted)),
        'balanced_accuracy': float(balanced_accuracy_score(actual, predicted)),
        'macro_f1': float(f1_score(actual, predicted, labels=labels,
                                 average='macro', zero_division=0)),
        'classification_report': classification_report(
            actual, predicted, labels=labels, target_names=list(names),
            output_dict=True, zero_division=0),
        'confusion_matrix': cm.tolist(), 'class_names': list(names),
        'samples': int(len(actual)),
    }
    if len(np.unique(actual)) == len(names):
        if len(names) == 2:
            result['roc_auc'] = float(roc_auc_score(actual, probabilities))
        else:
            result['roc_auc'] = float(roc_auc_score(
                actual, probabilities, labels=labels, multi_class='ovr', average='macro'))
    else:
        result['roc_auc'] = None
    metrics[name] = result
    print({k: result[k] for k in ('accuracy', 'balanced_accuracy', 'macro_f1', 'roc_auc')})
    print('Confusion matrix (filas=real, columnas=predicción):\n', cm)
    if plt is not None:
        plt.figure(figsize=(7, 5))
        sns.heatmap(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1),
                    annot=True, fmt='.1%', xticklabels=names, yticklabels=names)
        plt.title(name)
        plt.xlabel('Predicción')
        plt.ylabel('Real')
        plt.tight_layout()
        plt.show()
    return result

## 5. XGBoost — clasificación binaria normal / ataque

Optuna maximiza el macro-F1 medio de los tres folds internos de development.
Después se ajusta el candidato con todo development y se evalúa una sola vez
sobre el test externo, incluidas las conexiones cortas.

In [ ]:
def objective_xgb(trial):
    params = suggest_xgb_params(trial)
    fold_scores = []
    for fold_id, (inner_train, inner_val) in enumerate(inner_partitions):
        candidate = fit_classifier(X.iloc[inner_train], y_bin.iloc[inner_train], params)
        predicted = candidate.predict(X.iloc[inner_val])
        fold_scores.append(f1_score(
            y_bin.iloc[inner_val], predicted, average='macro', zero_division=0))
        trial.report(float(np.mean(fold_scores)), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr('fold_scores', [float(score) for score in fold_scores])
    return float(np.mean(fold_scores))


study_xgb = optuna.create_study(
    direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    study_name='xgboost_base_macro_f1')
study_xgb.optimize(objective_xgb, n_trials=OPTUNA_TRIALS_XGB)
print('Mejores hiperparámetros XGBoost:', study_xgb.best_params)
print('Mejor macro-F1 medio de CV interna:', study_xgb.best_value)
model_xgb = fit_classifier(X.iloc[development_idx], y_bin.iloc[development_idx],
                           study_xgb.best_params)
xgb_pred = model_xgb.predict(X.iloc[test_idx])
evaluate('XGBoost', y_bin.iloc[test_idx], xgb_pred,
         model_xgb.predict_proba(X.iloc[test_idx])[:, 1], BINARY_NAMES)
print('Accuracy development:', accuracy_score(
    y_bin.iloc[development_idx], model_xgb.predict(X.iloc[development_idx])))

# Detección de DoS por captura.
tabla_captura = (pd.DataFrame({
        'capture': df_raw.iloc[test_idx]['_capture'].to_numpy(),
        'real': y_bin.iloc[test_idx].to_numpy(),
        'pred': np.asarray(xgb_pred)})
    .query('real == 1')
    .groupby('capture')['pred']
    .agg(frames='size', detectados='sum'))
tabla_captura['recall'] = tabla_captura['detectados'] / tabla_captura['frames']
print('\nDetección de DoS por captura (XGBoost):')
print(tabla_captura)

importance = pd.Series(model_xgb.feature_importances_, index=X.columns).sort_values(ascending=False)
print('\nTop features:\n', importance.head(15))

## 6. LSTM Autoencoder — normal / ataque

Cada fold ajusta su propio scaler usando solo su train interno. Optuna
selecciona arquitectura, optimización y percentil del umbral mediante el
macro-F1 medio de los tres folds. Test se usa únicamente al final.


In [ ]:
X_mqtt = X.copy()

class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, num_features, hidden_dim=32):
        super().__init__()
        self.seq_len = seq_len
        self.encoder_lstm = nn.LSTM(num_features, hidden_dim, batch_first=True)
        self.decoder_lstm = nn.LSTM(hidden_dim, num_features, batch_first=True)

    def forward(self, x):
        _, (hidden, _) = self.encoder_lstm(x)
        latent = hidden.permute(1, 0, 2).repeat(1, self.seq_len, 1)
        output, _ = self.decoder_lstm(latent)
        return output


def train_autoencoder(train_x, params):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    model = LSTMAutoencoder(SEQ_LEN, train_x.shape[2],
                            hidden_dim=params['hidden_dim']).to(DEVICE)
    loader = DataLoader(TensorDataset(torch.from_numpy(train_x)),
                        batch_size=params['batch_size'], shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'],
                                 weight_decay=params['weight_decay'])
    model.train()
    for _ in range(params['epochs']):
        for (inputs,) in loader:
            inputs = inputs.to(DEVICE)
            optimizer.zero_grad()
            loss = ((model(inputs) - inputs) ** 2).mean()
            loss.backward()
            optimizer.step()
    return model


def reconstruction_errors(model, xs, batch_size):
    model.eval()
    errors = []
    loader = DataLoader(TensorDataset(torch.from_numpy(xs)),
                        batch_size=batch_size, shuffle=False, num_workers=0)
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            errors.append(((model(batch) - batch) ** 2).mean(dim=(1, 2)).cpu().numpy())
    return np.concatenate(errors)


def scaled_sequences(fitted_scaler, rows):
    flat_rows = rows.reshape(-1)
    xs = fitted_scaler.transform(X_mqtt.iloc[flat_rows]).astype(np.float32)
    xs = xs.reshape(len(rows), SEQ_LEN, X_mqtt.shape[1])
    binary = (y.to_numpy()[rows] != normal_idx).any(axis=1).astype(int)
    return xs, binary


def objective_lstm(trial):
    params = {
        'hidden_dim': trial.suggest_categorical('hidden_dim', [16, 32, 64, 128]),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 3e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-8, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [256, 512, 1024]),
        'epochs': trial.suggest_int('epochs', 5, 20, step=5),
        'threshold_percentile': trial.suggest_float('threshold_percentile', 90.0, 99.0),
    }
    fold_scores = []
    for fold_id, (inner_train, inner_val) in enumerate(inner_partitions):
        fold_scaler = StandardScaler().fit(X_mqtt.iloc[inner_train])
        train_data = scaled_sequences(fold_scaler, sequence_indices(inner_train))
        val_data = scaled_sequences(fold_scaler, sequence_indices(inner_val))
        train_normal = train_data[0][train_data[1] == 0]
        if not len(train_normal):
            raise ValueError(f'Fold interno {fold_id} sin ventanas normales de train')
        candidate = train_autoencoder(train_normal, params)
        normal_errors = reconstruction_errors(
            candidate, train_normal, params['batch_size'])
        candidate_threshold = float(np.percentile(
            normal_errors, params['threshold_percentile']))
        val_errors = reconstruction_errors(
            candidate, val_data[0], params['batch_size'])
        predicted = (val_errors > candidate_threshold).astype(int)
        fold_scores.append(f1_score(
            val_data[1], predicted, average='macro', zero_division=0))
        trial.report(float(np.mean(fold_scores)), step=fold_id)
        del candidate
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr('fold_scores', [float(score) for score in fold_scores])
    return float(np.mean(fold_scores))


study_lstm = optuna.create_study(
    direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    study_name='lstm_autoencoder_macro_f1')
study_lstm.optimize(objective_lstm, n_trials=OPTUNA_TRIALS_LSTM)
best_lstm_params = dict(study_lstm.best_params)
print('Mejores hiperparámetros LSTM:', best_lstm_params)
print('Mejor macro-F1 medio de CV interna:', study_lstm.best_value)
scaler = StandardScaler().fit(X_mqtt.iloc[development_idx])
window_rows = {
    'development': sequence_indices(development_idx),
    'test': sequence_indices(test_idx),
}
sequences = {name: scaled_sequences(scaler, rows)
             for name, rows in window_rows.items()}
development_normal = sequences['development'][0][sequences['development'][1] == 0]
if not len(development_normal):
    raise ValueError('No hay ventanas normales en development')
model_ae = train_autoencoder(development_normal, best_lstm_params)
mse_by_partition = {
    name: reconstruction_errors(model_ae, xs, best_lstm_params['batch_size'])
    for name, (xs, _) in sequences.items()}
development_normal_mse = mse_by_partition['development'][
    sequences['development'][1] == 0]
threshold = float(np.percentile(
    development_normal_mse, best_lstm_params['threshold_percentile']))
recon_errors = mse_by_partition['test']
true_labels = sequences['test'][1]
predicted = (recon_errors > threshold).astype(int)
print(f'Umbral calibrado con normal de development: {threshold:.6f}')
evaluate('LSTM', true_labels, predicted, recon_errors, ['normal', 'ataque'])


## 7. Híbrido LSTM + XGBoost — normal / ataque

El MSE de development se genera out-of-fold: cada conexión es reconstruida por
un autoencoder que no la vio al entrenar. Optuna ajusta el XGBoost híbrido con
3-fold grouped CV y el test externo nunca participa en los trials ni en el fit.

In [ ]:
def build_oof_hybrid_features():
    parts = []
    for fold_id, (inner_train, inner_val) in enumerate(inner_partitions):
        fold_scaler = StandardScaler().fit(X_mqtt.iloc[inner_train])
        train_data = scaled_sequences(fold_scaler, sequence_indices(inner_train))
        val_windows = sequence_indices(inner_val)
        val_data = scaled_sequences(fold_scaler, val_windows)
        train_normal = train_data[0][train_data[1] == 0]
        candidate = train_autoencoder(train_normal, best_lstm_params)
        val_mse = reconstruction_errors(
            candidate, val_data[0], best_lstm_params['batch_size'])
        ends = val_windows[:, -1]
        part = X.iloc[ends].copy().reset_index(drop=True)
        part['fe_lstm_mse'] = val_mse
        part['_row'] = ends
        parts.append(part)
        del candidate
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    combined = pd.concat(parts, ignore_index=True).sort_values('_row', kind='stable')
    rows = combined.pop('_row').to_numpy(dtype=int)
    return combined.reset_index(drop=True), rows


hybrid_development, hybrid_development_rows = build_oof_hybrid_features()
hybrid_test_rows = window_rows['test'][:, -1]
hybrid_test = X.iloc[hybrid_test_rows].copy().reset_index(drop=True)
hybrid_test['fe_lstm_mse'] = mse_by_partition['test']
hybrid = {'development': hybrid_development, 'test': hybrid_test}
hybrid_labels = {
    'development': y_bin.iloc[hybrid_development_rows].reset_index(drop=True),
    'test': y_bin.iloc[hybrid_test_rows].reset_index(drop=True),
}
hybrid_groups = connection_ids[hybrid_development_rows]
hybrid_splitter = StratifiedGroupKFold(
    n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
hybrid_inner_partitions = list(hybrid_splitter.split(
    hybrid_development, hybrid_labels['development'], groups=hybrid_groups))


def objective_hybrid(trial):
    params = suggest_xgb_params(trial)
    fold_scores = []
    for fold_id, (inner_train, inner_val) in enumerate(hybrid_inner_partitions):
        candidate = fit_classifier(
            hybrid_development.iloc[inner_train],
            hybrid_labels['development'].iloc[inner_train], params)
        predicted = candidate.predict(hybrid_development.iloc[inner_val])
        fold_scores.append(f1_score(
            hybrid_labels['development'].iloc[inner_val], predicted,
            average='macro', zero_division=0))
        trial.report(float(np.mean(fold_scores)), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr('fold_scores', [float(score) for score in fold_scores])
    return float(np.mean(fold_scores))


study_hybrid = optuna.create_study(
    direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    study_name='xgboost_hybrid_macro_f1')
study_hybrid.optimize(objective_hybrid, n_trials=OPTUNA_TRIALS_HYBRID)
print('Mejores hiperparámetros híbridos:', study_hybrid.best_params)
print('Mejor macro-F1 medio de CV interna:', study_hybrid.best_value)
model_hybrid = fit_classifier(hybrid['development'], hybrid_labels['development'],
                              study_hybrid.best_params)
hybrid_pred = model_hybrid.predict(hybrid['test'])
evaluate('LSTM + XGBoost', hybrid_labels['test'], hybrid_pred,
         model_hybrid.predict_proba(hybrid['test'])[:, 1], BINARY_NAMES)
# Comparación justa: XGBoost sobre exactamente los mismos frames.
ends = hybrid_test_rows
evaluate('XGBoost (frames con ventana)', y_bin.iloc[ends], model_xgb.predict(X.iloc[ends]),
         model_xgb.predict_proba(X.iloc[ends])[:, 1], BINARY_NAMES)

## 8. Comparación de medidas

Los tres modelos resuelven la misma tarea binaria (normal / DoS). En el contrato
exportado, la etiqueta `ataque` representa únicamente DoS. La accuracy
no basta con clases desbalanceadas: se reportan balanced accuracy, macro-F1 y ROC-AUC.

In [ ]:
resumen = pd.DataFrame([
    {'modelo': name, **{key: result[key] for key in (
        'samples', 'accuracy', 'balanced_accuracy', 'macro_f1', 'roc_auc')}}
    for name, result in metrics.items()
])
resumen


## 9. Exportación de los modelos evaluados

No se reentrena con test. Exportar y reemplazar todos los artefactos juntos.
`class_names` es `['normal', 'ataque']`, donde `ataque` es exclusivamente DoS,
y `normal_idx` es 0. `optuna_studies.json` conserva los mejores parámetros y
todos los trials. El agente permite
elegir `xgboost`, `lstm` o `hybrid`. `metrics.json` conserva los resultados y
`evaluation_split.json` identifica las filas originales seleccionadas y sus particiones.

In [ ]:
OUT_DIR = '/kaggle/working/modelos_agente'
os.makedirs(OUT_DIR, exist_ok=True)
model_xgb.save_model(os.path.join(OUT_DIR, 'xgb_case1.ubj'))
model_hybrid.save_model(os.path.join(OUT_DIR, 'xgb_hybrid.ubj'))
model_ae.cpu().eval()
scripted_ae = torch.jit.script(model_ae)
scripted_ae.save(os.path.join(OUT_DIR, 'lstm_ae.pt'))
joblib.dump(scaler, os.path.join(OUT_DIR, 'scaler.joblib'))
joblib.dump(le_binary, os.path.join(OUT_DIR, 'le_target.joblib'))
def study_payload(study):
    return {
        'study_name': study.study_name,
        'direction': study.direction.name,
        'best_value': float(study.best_value),
        'best_params': study.best_params,
        'trials': [
            {'number': trial.number, 'state': trial.state.name,
             'value': None if trial.value is None else float(trial.value),
             'params': trial.params, 'user_attrs': trial.user_attrs}
            for trial in study.trials
        ],
    }


tuning_payload = {
    'seed': SEED, 'objective': 'mean_inner_3fold_macro_f1',
    'inner_cv': {'n_splits': INNER_SPLITS, 'group': 'connection',
                 'development': 'outer train + validation'},
    'test_used_during_tuning': False,
    'xgboost': study_payload(study_xgb),
    'lstm': study_payload(study_lstm),
    'hybrid': study_payload(study_hybrid),
}
with open(os.path.join(OUT_DIR, 'optuna_studies.json'), 'w') as handle:
    json.dump(tuning_payload, handle, indent=2, allow_nan=False)
config = {
    'dataset': 'MQTT_UAD/DoS.csv', 'task': 'dos_binary',
    'positive_class': 'DoS', 'feature_mode': 'mqtt', 'capture_filter': '',
    'seq_len': SEQ_LEN, 'window_size': SEQ_LEN + 1, 'window_per_flow': True,
    'nan_fill': NAN_FILL, 'threshold': threshold, 'normal_idx': 0,
    'class_names': list(BINARY_NAMES),
    'split_strategy': 'external_test_plus_inner_3fold_grouped_development',
    'hyperparameter_tuning': {
        'framework': 'optuna', 'objective': 'mean_inner_3fold_macro_f1',
        'inner_splits': INNER_SPLITS, 'group': 'connection',
        'artifact': 'optuna_studies.json', 'test_used_during_tuning': False,
        'best_params': {
            'xgboost': study_xgb.best_params,
            'lstm': best_lstm_params,
            'hybrid': study_hybrid.best_params,
        },
    },
    'feature_columns_case1': list(X.columns),
    'feature_columns_lstm_raw': list(X.columns),
    'feature_columns_hybrid': list(hybrid['development'].columns),
}
with open(os.path.join(OUT_DIR, 'pipeline_config.json'), 'w') as handle:
    json.dump(config, handle, indent=2)
with open(os.path.join(OUT_DIR, 'metrics.json'), 'w') as handle:
    json.dump(metrics, handle, indent=2, allow_nan=False)
split_manifest = {
    'seed': SEED, 'selection': 'IPv4/TCP con endpoints completos',
    'development_from': ['train', 'val'], 'inner_splits': INNER_SPLITS,
    'partitions': {name: rows.tolist() for name, rows in partitions.items()},
    'source_rows': [
        {'capture': capture, 'original_row': int(row)}
        for capture, row in df_raw[['_capture', '_original_row']].itertuples(index=False, name=None)
    ],
}
with open(os.path.join(OUT_DIR, 'evaluation_split.json'), 'w') as handle:
    json.dump(split_manifest, handle, indent=2)
zip_path = '/kaggle/working/modelos_agente.zip'
# No incluir archivos viejos que hayan quedado de una ejecución anterior.
exported = ('xgb_case1.ubj', 'xgb_hybrid.ubj', 'lstm_ae.pt', 'scaler.joblib',
            'le_target.joblib', 'pipeline_config.json', 'metrics.json',
            'evaluation_split.json', 'optuna_studies.json')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in exported:
        archive.write(os.path.join(OUT_DIR, name), name)
print(f'Exportación completa: {zip_path}')